# §5.1 PCA-symmetry sweep — robustness of the `paper_tmlr_1` §7.2 local-conservativity finding

## What this notebook tests

`paper_tmlr_1` §7.2 reports that all three tested architectures (pretrained GPT-2 small, scale- and data-matched 8M attention baseline, scalar-potential probe SPLM) pass the velocity-aware Jacobian-symmetry test with max gap ≤ 0.079 — but the test is conducted at **PCA-16**. The §5.1 PCA-symmetry sweep, pre-registered in `docs/SP_HSPLM_Stage_2_q9e_n_verdict_and_structural_reexamination.md` in the main `semsimula` repository, asks whether the local-conservativity finding is robust under an increase in the PCA projection dimension or whether it is an artifact of aggressive dimensionality reduction.

## Decision rule

| Outcome | Max-layer gap at largest tested PCA-k | Paper-edit implication |
|---|---|---|
| **H2 REJECTED** | ≤ 0.10 | §7.2 stands; optional 1-paragraph robustness footnote before submission. |
| **H2 PARTIAL** | (0.10, 0.20] | One-paragraph scope-of-claim sharpening to §7.2. |
| **H2 CONFIRMED** | > 0.20 | Substantive rewrite of §7.2 framing and the introduction's contribution claims. The §8 three-way separator headline ($R^{2} = 0.949 / 0.56 / 0.45$) is unaffected and in fact *strengthens* under this reading. |

The default sweep is PCA-16 vs PCA-32 (the conservative 2-point sweep). PCA-64 is available as an optional third point but is over-parameterised at the 1,314-triplet corpus size and relies on the ridge regulariser built into `jacobian_symmetry.py`.

## Architectures covered

1. **Pretrained GPT-2 small** — public HF checkpoint, always run; this is the headline architecture for the verdict.
2. **Pythia-160M** — public HF checkpoint, always run (cross-architecture replication at the same scale used in `paper_tmlr_1` §6).
3. **MatchedGPT 8M** — scale- and data-matched attention baseline; local checkpoint, skipped if not present in `MATCHED_CKPT_GDRIVE_REL`.
4. **SPLM positive control** — scalar-potential probe; local checkpoint, skipped if not present in `SPLM_CKPT_GDRIVE_REL`.

The §5.1 verdict for the paper-edit cycle is determined by the GPT-2 row alone. The remaining three architectures, if present, sharpen the diagnosis but do not change the binary REJECTED / PARTIAL / CONFIRMED branch.

## Wall-clock budget

| Stage | Architecture(s) | Wall-clock on H100 |
|---|---|---|
| Extract GPT-2 small | gpt2 | ~3 min |
| Extract Pythia-160M | EleutherAI/pythia-160m | ~5 min |
| Extract MatchedGPT (if ckpt) | local | ~2 min |
| Extract SPLM (if ckpt) | local | ~2 min |
| Jacsym at PCA-16 + PCA-32 (per arch) | --- | ~3–6 min |
| Aggregate + verdict | --- | < 30 sec |

**Total: ≲ 30 min for GPT-2 + Pythia only; ≲ 1 h with all four architectures.**

## Reference

Decision rule, three-hypothesis framework, and paper-edit policy:
[`docs/SP_HSPLM_Stage_2_q9e_n_verdict_and_structural_reexamination.md`](https://github.com/dimitarpg13/semsimula/blob/main/docs/SP_HSPLM_Stage_2_q9e_n_verdict_and_structural_reexamination.md) §5.1 in the main `semsimula` repository.

## 0. Sweep configuration + Colab bootstrap

**Configurable knobs** (edit the next code cell):
- `SWEEP_PCA_KS` — list of PCA dimensions to test; default `[16, 32]`. Add `64` for the optional PCA-64 ridge variant.
- `RUN_GPT2` / `RUN_PYTHIA` — public-model toggles; always available.
- `RUN_MATCHED` / `RUN_SPLM` — local-checkpoint toggles; auto-skipped if the checkpoint files are absent at the listed GDrive paths.
- `MATCHED_CKPT_GDRIVE_REL` / `SPLM_CKPT_GDRIVE_REL` — relative paths under `/content/drive/MyDrive/`.

**Colab bootstrap (automatic).** When run on Google Colab, the next cell:
1. mounts your Google Drive at `/content/drive`,
2. shallow-clones (`--depth 1 --branch main`) the public `dimitarpg13/paper_tmlr_1` repo into `/content/paper_tmlr_1`,
3. routes all sweep outputs (jacsym `.npz` / figure / summary md / aggregate verdict / per-layer gap profile) to `/content/drive/MyDrive/paper_tmlr_1_pca_sweep/` — i.e. nothing important is written into the ephemeral `/content/paper_tmlr_1` clone,
4. installs `transformers`, `huggingface_hub`, `tokenizers`.

When run **locally** (off Colab), the notebook walks up from the CWD to find the standalone `paper_tmlr_1` repo root and writes to the in-repo path `notebooks/conservative_arch/scripts/results/`.

In [ ]:
# ===== Sweep configuration =====
SWEEP_PCA_KS = [16, 32]   # default conservative 2-point sweep; add 64 for the optional PCA-64 ridge variant
N_TEST_PER_DOMAIN = 2     # matches paper_tmlr_1 §7.2 train/test split
MAX_LEN = 64              # matches paper §7.2
SEED = 0

# Architecture toggles
RUN_GPT2    = True   # always available (public HF)
RUN_PYTHIA  = True   # always available (public HF)
RUN_MATCHED = True   # auto-skipped if MATCHED_CKPT_GDRIVE_REL absent
RUN_SPLM    = True   # auto-skipped if SPLM_CKPT_GDRIVE_REL absent

# Local-checkpoint paths (relative to /content/drive/MyDrive/); ignored if absent
MATCHED_CKPT_GDRIVE_REL = 'paper_tmlr_1_checkpoints/matched_baseline_ckpt_latest.pt'
SPLM_CKPT_GDRIVE_REL    = 'paper_tmlr_1_checkpoints/splm_em_ln_ckpt_latest.pt'

# ===== Source-of-truth repo + GDrive output dir =====
REPO_URL          = 'https://github.com/dimitarpg13/paper_tmlr_1.git'
REPO_BRANCH       = 'main'
COLAB_REPO_PATH   = '/content/paper_tmlr_1'
GDRIVE_OUT_REL    = 'paper_tmlr_1_pca_sweep'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}', flush=True)


def _sh(cmd):
    # Run a shell command, stream output, raise on non-zero exit.
    print(f'$ {cmd}', flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}', flush=True)

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    RESULTS_ROOT = GDRIVE_OUT
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    MATCHED_CKPT = Path('/content/drive/MyDrive') / MATCHED_CKPT_GDRIVE_REL
    SPLM_CKPT    = Path('/content/drive/MyDrive') / SPLM_CKPT_GDRIVE_REL

    _sh('pip install -q transformers huggingface_hub tokenizers')
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'notebooks').exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / 'notebooks').exists():
        raise RuntimeError(
            'Could not locate the paper_tmlr_1 repo root from the notebook CWD. '
            'cd into the standalone paper_tmlr_1 repo before launching the notebook.'
        )
    RESULTS_ROOT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scripts' / 'results'
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    MATCHED_CKPT = REPO_ROOT / 'checkpoints' / 'matched_baseline_ckpt_latest.pt'
    SPLM_CKPT    = REPO_ROOT / 'checkpoints' / 'splm_em_ln_ckpt_latest.pt'

CONS_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch'
SCRIPTS_DIR = CONS_DIR / 'scripts'
(CONS_DIR / 'results').mkdir(parents=True, exist_ok=True)

print('', flush=True)
print(f'REPO_ROOT     = {REPO_ROOT}', flush=True)
print(f'CONS_DIR      = {CONS_DIR}', flush=True)
print(f'SCRIPTS_DIR   = {SCRIPTS_DIR}', flush=True)
print(f'RESULTS_ROOT  = {RESULTS_ROOT}', flush=True)
print(f'SWEEP_PCA_KS  = {SWEEP_PCA_KS}', flush=True)
print(f'MATCHED_CKPT  = {MATCHED_CKPT}  (exists: {MATCHED_CKPT.exists()})', flush=True)
print(f'SPLM_CKPT     = {SPLM_CKPT}  (exists: {SPLM_CKPT.exists()})', flush=True)

## 1. Disable TF32 (numerical-precision invariant matches paper §7.2), set seeds, pick device

`paper_tmlr_1` §7.2's diagnostic is run in FP32 throughout. TF32 is disabled here to ensure the symmetric-restricted regression's R² values are bit-for-bit comparable to the paper's reported numbers.

In [ ]:
import torch
import numpy as np

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    cap = torch.cuda.get_device_capability()
    print(f'CUDA: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}  '
          f'mem={torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB', flush=True)
    if cap[0] < 8:
        print(f'  WARNING: compute capability < 8.0 ({cap}); designed for A100 (sm_80) or H100 (sm_90).', flush=True)
    print(f'  TF32 matmul = {torch.backends.cuda.matmul.allow_tf32}  (must be False)', flush=True)
    print(f'  TF32 cuDNN  = {torch.backends.cudnn.allow_tf32}  (must be False)', flush=True)
elif torch.backends.mps.is_available():
    device = 'mps'
    print('MPS device — TF32 toggles are CUDA-only.', flush=True)
else:
    device = 'cpu'
    print('CPU only — extraction is fast; jacsym at PCA-32 takes ~3-5 min per (arch, pca_k).', flush=True)
print(f'\ndevice = {device}', flush=True)

## 2. Helper: stream a subprocess's stdout live to the notebook

`stream_subprocess()` runs a command and prints each stdout line as it arrives, with `flush=True`, so progress is visible in real time — important for the jacsym runs which print per-layer R² values as they complete.

In [ ]:
import time, subprocess, shlex

def stream_subprocess(cmd, cwd=None, env=None, label=None):
    # Run `cmd` (list or str), stream stdout/stderr live to the notebook.
    # Returns (returncode, elapsed_sec).
    if isinstance(cmd, str):
        cmd_list = shlex.split(cmd)
    else:
        cmd_list = [str(c) for c in cmd]
    if label is None:
        label = Path(cmd_list[0]).name
    cmd_repr = ' '.join(shlex.quote(c) for c in cmd_list)
    print(f'\n[{label}] $ {cmd_repr}', flush=True)
    if cwd is not None:
        print(f'[{label}]   cwd = {cwd}', flush=True)
    t0 = time.time()
    p = subprocess.Popen(
        cmd_list, cwd=str(cwd) if cwd is not None else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, universal_newlines=True, text=True,
    )
    assert p.stdout is not None
    for line in p.stdout:
        sys.stdout.write(f'[{label}] {line}')
        sys.stdout.flush()
    rc = p.wait()
    elapsed = time.time() - t0
    print(f'[{label}] exit={rc}  elapsed={elapsed:.1f}s', flush=True)
    if rc != 0:
        raise RuntimeError(f'[{label}] command failed with exit code {rc}')
    return rc, elapsed

## 3. Stage 1 — Extract pretrained GPT-2 small trajectories

50 sentences × ≤64 tokens × 13 hidden states (input embedding + 12 layers).
Output: `notebooks/conservative_arch/results/gpt2_baseline.trajectories.pkl` (in-tree; copied to GDrive at Stage 5).

In [ ]:
PER_ARCH_PICKLES = {}

if RUN_GPT2:
    gpt2_out = CONS_DIR / 'results' / 'gpt2_baseline.trajectories.pkl'
    stream_subprocess(
        [sys.executable, 'extract_gpt2_baseline.py',
         '--model', 'gpt2',
         '--out',   str(gpt2_out),
         '--n_test_per_domain', str(N_TEST_PER_DOMAIN),
         '--max_len', str(MAX_LEN),
         '--seed', str(SEED)],
        cwd=CONS_DIR, label='extract-gpt2',
    )
    PER_ARCH_PICKLES['gpt2'] = gpt2_out
    assert gpt2_out.exists(), f'missing: {gpt2_out}'
    print(f'gpt2 trajectories: {gpt2_out}  size={gpt2_out.stat().st_size / 1e6:.1f} MB', flush=True)

## 4. Stage 2 — Extract Pythia-160M trajectories (cross-architecture replication)

Pythia-160M has the same depth as GPT-2 small (L=12) but uses rotary positional encoding and was trained on The Pile rather than WebText. A consistent PCA-sweep verdict across the two architectures rules out architecture-specific quirks as the source of the §7.2 finding.

We pass `--out` explicitly to avoid the slash in `EleutherAI/pythia-160m` from creating a subdirectory.

In [ ]:
if RUN_PYTHIA:
    pythia_out = CONS_DIR / 'results' / 'pythia-160m_baseline.trajectories.pkl'
    stream_subprocess(
        [sys.executable, 'extract_gpt2_baseline.py',
         '--model', 'EleutherAI/pythia-160m',
         '--out',   str(pythia_out),
         '--n_test_per_domain', str(N_TEST_PER_DOMAIN),
         '--max_len', str(MAX_LEN),
         '--seed', str(SEED)],
        cwd=CONS_DIR, label='extract-pythia',
    )
    PER_ARCH_PICKLES['pythia'] = pythia_out
    assert pythia_out.exists(), f'missing: {pythia_out}'
    print(f'pythia trajectories: {pythia_out}  size={pythia_out.stat().st_size / 1e6:.1f} MB', flush=True)

## 5. Stage 3 (optional) — Extract MatchedGPT 8M and SPLM positive-control trajectories

Both are gated on the corresponding checkpoint files being present under `/content/drive/MyDrive/`. To enable, train the checkpoints first using `notebooks/conservative_arch/train_matched.py` and `notebooks/conservative_arch/train_splm.py` (or `notebooks/conservative_arch/ln_damping_sweep/train_splm_em_ln.py` for the leak-corrected SPLM) and upload the resulting `.pt` files to the paths configured above.

If absent, the cell prints a skip notice and the sweep proceeds with the available architectures.

In [ ]:
# ---- MatchedGPT 8M ----
if RUN_MATCHED and MATCHED_CKPT.exists():
    matched_out = CONS_DIR / 'results' / 'matched_baseline.trajectories.pkl'
    stream_subprocess(
        [sys.executable, 'extract_matched_baseline.py',
         '--ckpt', str(MATCHED_CKPT),
         '--out',  str(matched_out),
         '--n_test_per_domain', str(N_TEST_PER_DOMAIN),
         '--max_len', str(MAX_LEN),
         '--seed', str(SEED)],
        cwd=CONS_DIR, label='extract-matched',
    )
    PER_ARCH_PICKLES['matched'] = matched_out
    assert matched_out.exists()
    print(f'matched trajectories: {matched_out}  size={matched_out.stat().st_size / 1e6:.1f} MB', flush=True)
else:
    print(f'[skip] matched: RUN_MATCHED={RUN_MATCHED}  '
          f'MATCHED_CKPT exists={MATCHED_CKPT.exists()}  ({MATCHED_CKPT})', flush=True)

# ---- SPLM positive control ----
if RUN_SPLM and SPLM_CKPT.exists():
    splm_out = CONS_DIR / 'results' / 'splm_em_ln.trajectories.pkl'
    stream_subprocess(
        [sys.executable, 'trajectory_extraction.py',
         '--ckpt', str(SPLM_CKPT),
         '--out',  str(splm_out),
         '--n_test_per_domain', str(N_TEST_PER_DOMAIN),
         '--max_len', str(MAX_LEN),
         '--seed', str(SEED)],
        cwd=CONS_DIR, label='extract-splm',
    )
    PER_ARCH_PICKLES['splm'] = splm_out
    assert splm_out.exists()
    print(f'splm trajectories: {splm_out}  size={splm_out.stat().st_size / 1e6:.1f} MB', flush=True)
else:
    print(f'[skip] splm: RUN_SPLM={RUN_SPLM}  '
          f'SPLM_CKPT exists={SPLM_CKPT.exists()}  ({SPLM_CKPT})', flush=True)

print(f'\nArchitectures to sweep: {sorted(PER_ARCH_PICKLES)}', flush=True)

## 6. Stage 4 — Run the velocity-aware Jacobian-symmetry test at every (architecture, PCA-k)

For each (architecture, PCA-k) the script prints `[jacsym] layer 0: ...` through `[jacsym] layer L-1: ...` to the notebook in real time, so you can see exactly where the sweep is at any moment. Each layer prints a line with the position-only and velocity-aware R² values; the symmetric-vs-unconstrained TEST gap (the headline of the §5.1 verdict) is computed downstream from the saved npz.

Run order: outer loop over architectures (so each architecture finishes both PCA-k values before the next architecture starts), inner loop over PCA-k. The GPT-2 result lands first since it determines the headline verdict.

In [ ]:
ALL_RESULTS = []
sweep_t0 = time.time()
total_runs = len(PER_ARCH_PICKLES) * len(SWEEP_PCA_KS)
run_idx = 0
for arch, pkl in PER_ARCH_PICKLES.items():
    for k in SWEEP_PCA_KS:
        run_idx += 1
        tag = f'{arch}_pca{k}'
        elapsed_so_far = time.time() - sweep_t0
        print(f'\n========== [{run_idx}/{total_runs}] [{arch} | PCA-{k}]  '
              f'(elapsed so far: {elapsed_so_far:.0f}s) ==========', flush=True)
        _, elapsed = stream_subprocess(
            [sys.executable, 'jacobian_symmetry.py',
             '--traj',   str(pkl),
             '--pca_k',  str(k),
             '--tag',    tag],
            cwd=CONS_DIR, label=f'jacsym/{arch}/pca{k}',
        )
        ALL_RESULTS.append({
            'arch': arch, 'pca_k': k, 'tag': tag,
            'elapsed_sec': elapsed,
            'npz': CONS_DIR / 'results' / f'splm_{tag}_jacsym_results.npz',
            'fig': CONS_DIR / 'results' / f'splm_{tag}_fig_jacsym.png',
            'md':  CONS_DIR / 'results' / f'splm_{tag}_jacsym_summary.md',
        })

sweep_elapsed = time.time() - sweep_t0
print(f'\n[sweep] completed {len(ALL_RESULTS)} runs in {sweep_elapsed:.0f}s '
      f'({sweep_elapsed/60:.1f} min)', flush=True)
for r in ALL_RESULTS:
    exists = r['npz'].exists()
    print(f'  - {r["arch"]} PCA-{r["pca_k"]}: {r["elapsed_sec"]:.1f}s  '
          f'-> {r["npz"].name}  exists={exists}', flush=True)

## 7. Stage 5 — Copy every jacsym artefact to GDrive

This makes the per-(architecture, PCA-k) results persistent across Colab sessions and feeds the aggregator in the next stage.

In [ ]:
ARTEFACTS_COPIED = 0
for r in ALL_RESULTS:
    for src in (r['npz'], r['fig'], r['md']):
        if src.exists():
            dst = RESULTS_ROOT / src.name
            shutil.copy(src, dst)
            ARTEFACTS_COPIED += 1
            print(f'copied: {src.name} -> {dst}', flush=True)
        else:
            print(f'WARNING: missing artefact (skipping copy): {src}', flush=True)
print(f'\n[sweep] copied {ARTEFACTS_COPIED} artefacts to {RESULTS_ROOT}', flush=True)

## 8. Stage 6 — Aggregate the per-(architecture, PCA-k) results and emit the §5.1 verdict

`scripts/aggregate_pca_sweep.py` reads every `splm_*_pca*_jacsym_results.npz` under `RESULTS_ROOT`, builds the per-layer gap-comparison table, plots the per-layer gap profile, and emits the REJECTED / PARTIAL / CONFIRMED classification per architecture and overall.

In [ ]:
stream_subprocess(
    [sys.executable, str(SCRIPTS_DIR / 'aggregate_pca_sweep.py'),
     '--in-dir',  str(RESULTS_ROOT),
     '--out-dir', str(RESULTS_ROOT),
     '--pca-ks'] + [str(k) for k in SWEEP_PCA_KS],
    cwd=SCRIPTS_DIR, label='aggregate',
)

## 9. Display the headline verdict, comparison table, and per-layer gap profile inline

In [ ]:
from IPython.display import Image, Markdown, display

verdict_path = RESULTS_ROOT / 'pca_sweep_verdict.md'
table_path   = RESULTS_ROOT / 'pca_sweep_table.md'
fig_path     = RESULTS_ROOT / 'pca_sweep_gap_profile.png'
summary_path = RESULTS_ROOT / 'pca_sweep_summary.json'

if verdict_path.exists():
    display(Markdown('## §5.1 verdict'))
    display(Markdown(verdict_path.read_text()))
if table_path.exists():
    display(Markdown('## Per-architecture comparison table'))
    display(Markdown(table_path.read_text()))
if fig_path.exists():
    display(Markdown('## Per-layer gap profile'))
    display(Image(filename=str(fig_path)))
if summary_path.exists():
    display(Markdown('## Machine-readable summary'))
    print(summary_path.read_text())

print(f'\nAll artefacts saved under: {RESULTS_ROOT}', flush=True)

## 10. CLI equivalents (for SLURM / nohup / headless launches)

```bash
cd notebooks/conservative_arch/
mkdir -p results

# 1. Extract trajectories.
python extract_gpt2_baseline.py --model gpt2 \
  --out results/gpt2_baseline.trajectories.pkl \
  --n_test_per_domain 2 --max_len 64 --seed 0
python extract_gpt2_baseline.py --model EleutherAI/pythia-160m \
  --out results/pythia-160m_baseline.trajectories.pkl \
  --n_test_per_domain 2 --max_len 64 --seed 0
# (matched / splm extractions only if the corresponding checkpoints are present)

# 2. Run the PCA-symmetry sweep.
for ARCH in gpt2 pythia; do
  for K in 16 32; do
    python jacobian_symmetry.py \
      --traj  results/${ARCH}_baseline.trajectories.pkl \
      --pca_k $K --tag ${ARCH}_pca${K}
  done
done

# 3. Aggregate the verdict.
python scripts/aggregate_pca_sweep.py \
  --in-dir results --out-dir results --pca-ks 16 32
```

Output paths (local CLI): `notebooks/conservative_arch/results/splm_<arch>_pca<k>_*.{npz,png,md}` plus `pca_sweep_table.md`, `pca_sweep_verdict.md`, `pca_sweep_gap_profile.png`, `pca_sweep_summary.json`.

On Colab: same filenames under `/content/drive/MyDrive/paper_tmlr_1_pca_sweep/`.